# 🖥️ OCR API Server — Colab Backend
> Chạy lần lượt từng cell. Sau cell cuối bạn sẽ có một URL public để kết nối từ frontend.

## Cell 1 — Cài vLLM

In [1]:
# ==================== Cài vLLM trên Colab T4 ====================
!pip uninstall -q vllm -y
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -q vllm --extra-index-url https://wheels.vllm.ai/nightly
print("✅ Cài vLLM xong!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 182.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 154.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 168.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3

## Cell 2 — Khởi động vLLM server (port 8008, internal)

In [1]:
import subprocess, threading, time
from IPython.display import clear_output

def start_vllm():
    cmd = [
        "vllm", "serve", "Qwen/Qwen3.6-35B-A3B",
        "--port", "8008",
        "--gpu-memory-utilization", "0.9",
        "--kv-cache-dtype", "fp8_e5m2",
        "--dtype", "bfloat16",
        "--max-model-len", "32768",
        "--max-num-seqs", "64",
        "--max-num-batched-tokens", "16384",
        "--enable-prefix-caching",
        "--enable-chunked-prefill",
        "--trust-remote-code",
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
    for line in process.stdout:
        print(line, end='')

thread = threading.Thread(target=start_vllm, daemon=True)
thread.start()
print("⌛ Đang load model Qwen3.6-35B-A3B... (~60-90s)")
time.sleep(75)
clear_output(wait=True)
print("✅ vLLM Server đang chạy tại localhost:8008")

✅ vLLM Server đang chạy tại localhost:8008


## Cell 3 — Cài thư viện API

In [1]:
!apt-get install -y -q poppler-utils
!pip install -q pdf2image Pillow
!pip install -q fastapi uvicorn[standard] python-multipart aiofiles sse-starlette
print("✅ Cài xong dependencies")

Reading package lists...
Building dependency tree...
Reading state information...
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
✅ Cài xong dependencies


## Cell 4 — Định nghĩa FastAPI app

In [6]:
import os, re, time, base64, asyncio, uuid, json
import requests as req_sync
from io import BytesIO
from pathlib import Path
from typing import Optional
from concurrent.futures import ThreadPoolExecutor, as_completed

from PIL import Image
from pdf2image import convert_from_bytes, convert_from_path

from fastapi import FastAPI, File, UploadFile, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse, StreamingResponse
from sse_starlette.sse import EventSourceResponse

# ─── Config ────────────────────────────────────────────────────────────────
VLLM_URL   = "http://localhost:8008/v1/chat/completions"
MODEL_NAME = "Qwen/Qwen3.6-35B-A3B"
MAX_WORKERS = 32
DPI = 300

# ─── App ────────────────────────────────────────────────────────────────────
app = FastAPI(title="OCR API", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ─── Helpers ────────────────────────────────────────────────────────────────
def encode_pil(img: Image.Image) -> str:
    buf = BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=95)
    return base64.b64encode(buf.getvalue()).decode()

def clean_output(text: str) -> str:
    text = re.sub(r"<thinking>.*?</thinking>", "", text, flags=re.DOTALL)
    text = re.sub(r"<thinking>.*", "", text, flags=re.DOTALL)
    return re.sub(r"</?thinking>", "", text).strip()

def load_images_from_bytes(data: bytes, filename: str):
    """Returns list of (page_num, PIL.Image)"""
    fname = filename.lower()
    if fname.endswith(".pdf"):
        pages = convert_from_bytes(data, dpi=DPI)
        return [(i + 1, p.convert("RGB")) for i, p in enumerate(pages)]
    else:
        img = Image.open(BytesIO(data)).convert("RGB")
        return [(1, img)]

def ocr_one_page(b64: str, page_num: int, total: int, fname: str):
    t0 = time.time()
    payload = {
        "model": MODEL_NAME,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "text", "text": (
                    f"Bạn là hệ thống OCR tiếng Việt chuyên nghiệp thực hiện nhiệm vụ "
                    f"rà soát văn bản từ hình ảnh hoặc các file scan về các cáo trạng cho VKS.\n"
                    f"File: {fname} — Trang {page_num}/{total}\n\n"
                    "Yêu cầu:\n"
                    "- Chỉ trả về nội dung OCR cuối cùng.\n"
                    "- KHÔNG giải thích, KHÔNG in ra suy nghĩ.\n"
                    "- Giữ nguyên cấu trúc văn bản, bảng biểu, số liệu.\n"
                    "- Không bịa số liệu hay dữ liệu.\n"
                    "- Suy luận hợp lý khi nội dung khó đọc, không bịa sai hoàn toàn.\n"
                    "- Output chỉ gồm text đúng cấu trúc."
                )},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}}
            ]
        }],
        "max_tokens": 4096,
        "temperature": 0.0,
        "repetition_penalty": 1.1,
        "frequency_penalty": 0.3,
        "chat_template_kwargs": {"enable_thinking": False}
    }
    try:
        r = req_sync.post(VLLM_URL, json=payload, timeout=300)
        data = r.json()
        elapsed = time.time() - t0
        if "choices" in data:
            text = clean_output(data["choices"][0]["message"]["content"])
            return {"page": page_num, "text": text, "time_s": round(elapsed, 2), "ok": True}
        return {"page": page_num, "text": str(data), "time_s": round(elapsed, 2), "ok": False}
    except Exception as e:
        return {"page": page_num, "text": str(e), "time_s": 0.0, "ok": False}

# ─── Routes ──────────────────────────────────────────────────────────────────

@app.get("/health")
def health():
    """Kiểm tra server còn sống không"""
    try:
        r = req_sync.get("http://localhost:8008/v1/models", timeout=5)
        models = r.json().get("data", [])
        return {"status": "ok", "vllm": "ready", "models": [m["id"] for m in models]}
    except Exception as e:
        return {"status": "ok", "vllm": "loading", "detail": str(e)}


@app.post("/ocr")
async def ocr_sync(file: UploadFile = File(...)):
    """
    Upload PDF hoặc ảnh, trả về JSON đầy đủ sau khi xử lý xong.
    Phù hợp cho file nhỏ hoặc khi không cần streaming.
    """
    data = await file.read()
    fname = file.filename or "upload"

    try:
        pages = load_images_from_bytes(data, fname)
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"Không đọc được file: {e}")

    total = len(pages)
    work = [(encode_pil(img), pnum, total, fname) for pnum, img in pages]
    t0 = time.time()
    results = []

    loop = asyncio.get_event_loop()
    with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, total)) as ex:
        futures = {ex.submit(ocr_one_page, *w): w for w in work}
        tasks = [loop.run_in_executor(None, lambda f=f: f.result()) for f in futures]
        for coro in asyncio.as_completed(tasks):
            res = await coro
            results.append(res)

    results.sort(key=lambda x: x["page"])
    total_time = round(time.time() - t0, 2)

    return {
        "filename": fname,
        "total_pages": total,
        "total_time_s": total_time,
        "pages": results
    }


@app.post("/ocr/stream")
async def ocr_stream(file: UploadFile = File(...)):
    """
    Streaming SSE — gửi từng trang khi xong.
    Frontend nhận event 'page' liên tục, event 'done' khi hoàn tất.

    Event format:
      data: {"type": "start",  "total_pages": N, "filename": "..."}
      data: {"type": "page",   "page": N, "text": "...", "time_s": X, "ok": true}
      data: {"type": "done",   "total_time_s": X}
      data: {"type": "error",  "detail": "..."}
    """
    data = await file.read()
    fname = file.filename or "upload"

    async def event_generator():
        try:
            pages = load_images_from_bytes(data, fname)
        except Exception as e:
            yield {"data": json.dumps({"type": "error", "detail": str(e)})}
            return

        total = len(pages)
        yield {"data": json.dumps({"type": "start", "total_pages": total, "filename": fname})}

        t0 = time.time()
        work = [(encode_pil(img), pnum, total, fname) for pnum, img in pages]
        queue: asyncio.Queue = asyncio.Queue()
        loop = asyncio.get_event_loop()

        def run_and_queue(args):
            result = ocr_one_page(*args)
            asyncio.run_coroutine_threadsafe(queue.put(result), loop)

        with ThreadPoolExecutor(max_workers=min(MAX_WORKERS, total)) as ex:
            for w in work:
                ex.submit(run_and_queue, w)

            received = 0
            while received < total:
                res = await queue.get()
                received += 1
                yield {"data": json.dumps({"type": "page", **res})}

        total_time = round(time.time() - t0, 2)
        yield {"data": json.dumps({"type": "done", "total_time_s": total_time})}

    return EventSourceResponse(event_generator())


## Cell 5 — Khởi động FastAPI server + tunnel ra ngoài

In [7]:
import threading, time, os, subprocess, re
import requests as req_sync
import uvicorn

API_PORT  = 8900
SUBDOMAIN = "vks-ocr-hvks"
FIXED_URL = f"https://{SUBDOMAIN}.loca.lt"

print("[1/5] start uvicorn ...", flush=True)

def run_api():
    uvicorn.run(app, host="0.0.0.0", port=API_PORT, log_level="warning")

threading.Thread(target=run_api, daemon=True).start()
time.sleep(8)

print("[2/5] check /health ...", flush=True)
try:
    r = req_sync.get(f"http://localhost:{API_PORT}/health", timeout=5)
    print("      status:", r.status_code, "body:", r.text[:200], flush=True)
except Exception as e:
    print("      ⚠️ health check failed:", e, flush=True)

print("[3/5] install localtunnel if missing ...", flush=True)
if subprocess.run("which lt", shell=True, capture_output=True).returncode != 0:
    subprocess.run("npm install -g localtunnel -q", shell=True, timeout=120)
print("      done", flush=True)

print(f"[4/5] start tunnel với subdomain cố định: {SUBDOMAIN} ...", flush=True)

def _try_tunnel():
    subprocess.run("pkill -9 -f 'lt --port' || true", shell=True, timeout=5)
    time.sleep(2)
    try: os.remove("tunnel_api.log")
    except: pass
    get_ipython().system_raw(
        f"lt --port {API_PORT} --subdomain {SUBDOMAIN} > tunnel_api.log 2>&1 &"
    )
    for _ in range(40):
        time.sleep(1)
        try:
            log = open("tunnel_api.log").read()
            m = re.search(r"https://[\w-]+\.loca\.lt", log)
            if m: return m.group(0), log
        except FileNotFoundError:
            pass
    try: log = open("tunnel_api.log").read()
    except: log = ""
    return None, log

clean_url = None
last_log = ""
for attempt in range(1, 4):
    print(f"      attempt {attempt}/3 ...", flush=True)
    url, last_log = _try_tunnel()
    if url == FIXED_URL:
        clean_url = url
        break
    if url:
        print(f"      ⚠️ nhận {url} thay vì {FIXED_URL}, retry ...", flush=True)
    else:
        print(f"      ⚠️ chưa nhận URL, retry ...", flush=True)

print("[5/5] kết quả", flush=True)
if clean_url:
    print(f"\n✅ URL CỐ ĐỊNH: {clean_url}")
    try:
        pw = req_sync.get("https://loca.lt/mytunnelpassword", timeout=10).text.strip()
        print(f"🔑 Tunnel password: {pw}")
    except: pass
    print(f"\n📋 Endpoints:")
    print(f"  GET  {clean_url}/health")
    print(f"  POST {clean_url}/ocr")
    print(f"  POST {clean_url}/ocr/stream")
else:
    print(f"❌ Không chiếm được subdomain '{SUBDOMAIN}' sau 3 lần thử.")
    print("Log cuối của localtunnel:")
    print(last_log)
    print(f"\n→ Subdomain có thể đang bị chiếm. Đổi biến SUBDOMAIN trong cell này rồi chạy lại.")


[1/5] start uvicorn ...
[2/5] check /health ...
      status: 200 body: {"status":"ok","vllm":"ready","models":["Qwen/Qwen3.6-35B-A3B"]}
[3/5] install localtunnel if missing ...
      done
[4/5] start tunnel với subdomain cố định: vks-ocr-hvks ...
      attempt 1/3 ...
[5/5] kết quả

✅ URL CỐ ĐỊNH: https://vks-ocr-hvks.loca.lt
🔑 Tunnel password: 35.192.118.83

📋 Endpoints:
  GET  https://vks-ocr-hvks.loca.lt/health
  POST https://vks-ocr-hvks.loca.lt/ocr
  POST https://vks-ocr-hvks.loca.lt/ocr/stream


## Cell 5b — Keep-alive (giữ Colab + tunnel luôn sống)
> Chạy ngay sau Cell 5. Kết hợp JS chống idle UI + self-ping `/health` cho uvicorn và tunnel. Chỉ dừng khi bạn tắt runtime.

In [8]:
# _COLAB_KEEPALIVE_MARKER_
# ==================== Keep-alive: JS anti-idle + self-ping ====================
from IPython.display import display, Javascript
import threading, time, requests as _ka_req

# 1) JS: click nút reconnect + fake mouse move trong tab browser (mỗi 45s)
display(Javascript(r"""
(function(){
  if (window.__colabKeepAlive) return;
  window.__colabKeepAlive = true;
  function tick(){
    try {
      document.querySelectorAll('colab-connect-button').forEach(b => {
        const shadow = b.shadowRoot;
        if (shadow) { const btn = shadow.querySelector('#connect'); if (btn) btn.click(); }
      });
      document.querySelectorAll('paper-icon-button[icon="notification:sync"]').forEach(b => b.click());
      document.dispatchEvent(new MouseEvent('mousemove', {clientX: 1, clientY: 1, bubbles: true}));
      document.dispatchEvent(new KeyboardEvent('keydown', {key: 'Shift', bubbles: true}));
    } catch(e) {}
  }
  setInterval(tick, 45*1000);
  console.log('[keepalive] JS registered');
})();
"""))

# 2) Python: self-ping /health mỗi 60s — giữ uvicorn + localtunnel active
_KA_STOP = threading.Event()
def _ka_loop():
    tunnel = None
    try:
        tunnel = clean_url  # noqa: F821 — biến từ Cell 5
    except NameError:
        pass
    fails = 0
    while not _KA_STOP.is_set():
        try:
            _ka_req.get('http://localhost:8900/health', timeout=5)
        except Exception:
            pass
        if tunnel:
            try:
                _ka_req.get(tunnel + '/health',
                            headers={'bypass-tunnel-reminder': 'true'},
                            timeout=10)
                fails = 0
            except Exception:
                fails += 1
        _KA_STOP.wait(60)

# Dừng thread cũ nếu có rồi khởi động mới (idempotent khi rerun cell)
if globals().get('_ka_thread') and _ka_thread.is_alive():
    _KA_STOP.set(); _ka_thread.join(timeout=2); _KA_STOP = threading.Event()
_ka_thread = threading.Thread(target=_ka_loop, daemon=True, name='colab-keepalive')
_ka_thread.start()

print("✅ Keep-alive bật: JS click (45s) + self-ping local & tunnel (60s)")
print("   → Chỉ dừng khi bạn tắt runtime Colab.")


<IPython.core.display.Javascript object>

✅ Keep-alive bật: JS click (45s) + self-ping local & tunnel (60s)
   → Chỉ dừng khi bạn tắt runtime Colab.


## Cell 6 — Test nhanh (tuỳ chọn)

In [9]:
# Test health endpoint (diagnostic)
import requests as req_sync

BASE = 'http://localhost:8900'

r = req_sync.get(f'{BASE}/health', timeout=10)
print('status:', r.status_code)
print('content-type:', r.headers.get('content-type'))
print('body (first 500 chars):', repr(r.text[:500]))
try:
    print('JSON:', r.json())
except Exception as e:
    print('JSON parse failed:', e)

# Nếu body rỗng hoặc không JSON:
#   - Kiểm tra uvicorn có thực sự listen port 9000 chưa: !ss -lntp | grep 8080
#   - Xem log uvicorn: chạy lại Cell 5 với log_level="info"
#   - Thử cổng khác nếu 8080 bị chiếm


status: 200
content-type: application/json
body (first 500 chars): '{"status":"ok","vllm":"ready","models":["Qwen/Qwen3.6-35B-A3B"]}'
JSON: {'status': 'ok', 'vllm': 'ready', 'models': ['Qwen/Qwen3.6-35B-A3B']}
